In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [2]:
book_url = "https://books.toscrape.com/"

headers = {"User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36"}

book_response = requests.get(book_url,headers=headers)

if book_response.status_code == 200:
    print("Connection successful!")
else:
    print(f"Connection failed: {book_response.status_code}")


Connection successful!


In [4]:
print(book_response.content[:1000])

b'<!DOCTYPE html>\n<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->\n<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->\n<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->\n<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->\n    <head>\n        <title>\n    All products | Books to Scrape - Sandbox\n</title>\n\n        <meta http-equiv="content-type" content="text/html; charset=UTF-8" />\n        <meta name="created" content="24th Jun 2016 09:29" />\n        <meta name="description" content="" />\n        <meta name="viewport" content="width=device-width" />\n        <meta name="robots" content="NOARCHIVE,NOCACHE" />\n\n        <!-- Le HTML5 shim, for IE6-8 support of HTML elements -->\n        <!--[if lt IE 9]>\n        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>\n        <![endif]-->\n\n        \n            <link rel="shortcut icon" hre

In [5]:
book_soup = BeautifulSoup(book_response.content,"html.parser")

type(book_soup)

bs4.BeautifulSoup

In [6]:
print(book_soup.prettify()[:1000])

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!-->
<html class="no-js" lang="en-us">
 <!--<![endif]-->
 <head>
  <title>
   All products | Books to Scrape - Sandbox
  </title>
  <meta content="text/html; charset=utf-8" http-equiv="content-type"/>
  <meta content="24th Jun 2016 09:29" name="created"/>
  <meta content="" name="description"/>
  <meta content="width=device-width" name="viewport"/>
  <meta content="NOARCHIVE,NOCACHE" name="robots"/>
  <!-- Le HTML5 shim, for IE6-8 support of HTML elements -->
  <!--[if lt IE 9]>
        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>
        <![endif]-->
  <link href="static/oscar/favicon.ico" rel="shortcut icon"/>
  <link href="static/oscar/css/styles.css" rel="stylesheet" type="tex

In [13]:
rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
temp_books_list = []

for page in range(1, 51):
    page_url = "https://books.toscrape.com/catalogue/page-" + str(page) + ".html"
    response = requests.get(page_url, headers=headers)
    if response.status_code != 200:
        continue
        
    soup = BeautifulSoup(response.content, "html.parser")
    books = soup.find_all('article', class_='product_pod')
    
    for book in books:
        price_text = book.find('p', class_='price_color').text
        price_num = float(price_text.replace('£', ''))
        
        rating_class = book.find('p', class_='star-rating')['class'][1]
        rating_num = rating_map.get(rating_class, 0)
        
        if rating_num >= 4.0 and price_num <= 20.0:
            raw_href = book.h3.a['href']
            clean_href = raw_href.replace('../', '')
            detail_url = "https://books.toscrape.com/catalogue/" + clean_href
            
            temp_books_list.append({
                "Link": detail_url,
                "Price": price_num,
                "Rating": rating_num
            })

In [14]:
len(temp_books_list)

75

In [18]:
all_filtered_books = []

for item in temp_books_list:
    detail_url = item["Link"]
    
    try:
        detail_response = requests.get(detail_url, headers=headers)
        if detail_response.status_code != 200:
            continue
            
        detail_soup = BeautifulSoup(detail_response.content, "html.parser")
        
        title = detail_soup.find('div', class_='product_main').h1.text
        
        breadcrumb = detail_soup.find('ul', class_='breadcrumb')
        genre = breadcrumb.find_all('li')[2].text.strip() if breadcrumb else "Unknown"
        
        desc_tag = detail_soup.find('div', id='product_description')
        description = desc_tag.find_next_sibling('p').text if desc_tag else "No description"
        
        table = detail_soup.find('table', class_='table-striped')
        upc = "Unknown"
        availability = "Unknown"
        
        if table:
            rows = table.find_all('tr')
            for row in rows:
                header = row.th.text.strip()
                value = row.td.text.strip()
                if header == "UPC":
                    upc = value
                elif header == "Availability":
                    availability = value
                    
        all_filtered_books.append({
            "UPC": upc,
            "Title": title,
            "Price (£)": item["Price"],
            "Rating": item["Rating"],
            "Genre": genre,
            "Availability": availability,
            "Description": description
        })
    except:
        pass

In [19]:
book_df = pd.DataFrame(all_filtered_books)
type(book_df)

pandas.core.frame.DataFrame

In [20]:
book_df.head()

,UPC,Title,Price (£),Rating,Genre,Availability,Description
0,ce6396b0f23f6ecc,Set Me Free,17.46,5,Young Adult,In stock (19 available),Aaron Ledbetter’s future had been planned out ...
1,6258a1f6a6dcfe50,The Four Agreements: A Practical Guide to Pers...,17.66,5,Spirituality,In stock (18 available),"In The Four Agreements, don Miguel Ruiz reveal..."
2,6be3beb0793a53e7,Sophie's World,15.94,5,Philosophy,In stock (18 available),A page-turning novel that is also an explorati...
3,657fe5ead67a7767,Untitled Collection: Sabbath Poems 2014,14.27,4,Poetry,In stock (16 available),"More than thirty-five years ago, when the weat..."
4,51653ef291ab7ddc,This One Summer,19.49,4,Sequential Art,In stock (16 available),"Every summer, Rose goes with her mom and dad t..."
